In [0]:
# Databricks notebook source
from delta.tables import DeltaTable
from pyspark.sql import functions as F

STORAGE = "stecomlakedev"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"
GOLD = f"abfss://gold@{STORAGE}.dfs.core.windows.net"

DIM_CUSTOMER = f"{GOLD}/dim_customer"

# COMMAND ----------
src = (spark.read.format("delta").load(f"{SILVER}/customers")
    .select(
        F.col("customer_unique_id").alias("customer_id"),
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state"))
    .dropDuplicates(["customer_id"]))

def stamp(df, sk_offset=0):
    return (df
        .withColumn("customer_sk", F.monotonically_increasing_id() + F.lit(sk_offset))
        .withColumn("effective_from", F.current_timestamp())
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True)))

# Checks the storage path rather than cached session metadata: isDeltaTable can
# answer from a stale cache after the table is deleted underneath Spark.
def table_exists(path):
    try:
        dbutils.fs.ls(f"{path}/_delta_log")
        return True
    except Exception:
        return False

# COMMAND ----------
if not table_exists(DIM_CUSTOMER):
    stamp(src).write.format("delta").save(DIM_CUSTOMER)
    print("initial load")
else:
    dim = DeltaTable.forPath(spark, DIM_CUSTOMER)
    current = dim.toDF().filter("is_current")

    changes = (src.alias("s")
        .join(current.alias("d"), "customer_id")
        .filter("s.city <> d.city OR s.state <> d.state")
        .select("s.*"))

    dim.alias("d").merge(
            changes.alias("c"),
            "d.customer_id = c.customer_id AND d.is_current = true") \
        .whenMatchedUpdate(set={"is_current": "false",
                                "effective_to": "current_timestamp()"}) \
        .execute()

    new_customers = src.join(dim.toDF().select("customer_id").distinct(),
                             "customer_id", "left_anti")

    # monotonically_increasing_id restarts near zero on every write, so an offset
    # past the current maximum is what keeps surrogate keys unique across appends.
    max_sk = dim.toDF().agg(
        F.coalesce(F.max("customer_sk"), F.lit(0)).alias("m")).collect()[0]["m"]

    to_insert = changes.unionByName(new_customers)
    if to_insert.count() > 0:
        stamp(to_insert, max_sk + 1).write.format("delta").mode("append").save(DIM_CUSTOMER)
    print(f"expired+inserted: {to_insert.count()}")

# COMMAND ----------
# The unknown member lets facts that cannot resolve a customer keep their measures
# instead of being dropped, so totals still reconcile.
dim_cust = spark.read.format("delta").load(DIM_CUSTOMER)

if dim_cust.filter("customer_sk = -1").count() == 0:
    spark.createDataFrame(
        [("UNKNOWN", "Unknown", "XX", -1, None, None, True)],
        dim_cust.schema
    ).write.format("delta").mode("append").save(DIM_CUSTOMER)

print(f"dim_customer: {spark.read.format('delta').load(DIM_CUSTOMER).count()} rows")

# COMMAND ----------
# SCD1: product attribute changes are corrections, not events, so no history.
products = spark.read.format("delta").load(f"{SILVER}/products")

dim_product = (products
    .select("product_id", "category_name",
            F.col("product_weight_g").alias("weight_g"),
            F.col("product_volume_cm3").alias("volume_cm3"))
    .withColumn("size_band",
        F.when(F.col("volume_cm3") < 5000, "small")
         .when(F.col("volume_cm3") < 30000, "medium")
         .otherwise("large")))

(dim_product.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(f"{GOLD}/dim_product"))

print(f"dim_product: {spark.read.format('delta').load(f'{GOLD}/dim_product').count()} rows")

# COMMAND ----------
dim_date = (spark.sql(
        "SELECT explode(sequence(to_date('2016-01-01'), to_date('2027-12-31'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
    .select("date_key", "date", "year", "quarter", "month", "month_name",
            "day_of_month", "day_of_week", "day_name", "week_of_year", "is_weekend"))

(dim_date.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(f"{GOLD}/dim_date"))

print(f"dim_date: {spark.read.format('delta').load(f'{GOLD}/dim_date').count()} rows")

# COMMAND ----------
d = spark.read.format("delta").load(DIM_CUSTOMER)
assert d.filter("is_current").groupBy("customer_id").count().filter("count > 1").count() == 0
assert d.groupBy("customer_sk").count().filter("count > 1").count() == 0
print("dimension invariants OK")